# Mlp final

In [1]:
# notebooks/neural_recommender.ipynb
import torch
import torch.nn as nn

class ContentBasedMLP(nn.Module):
    """MLP que combina embedding de usuário com features do item."""

    def __init__(
        self,
        n_users: int,
        n_categories: int,
        embedding_dim: int = 32,
        hidden_dims: list[int] = [128, 64, 32],
    ) -> None:
        super().__init__()
        # Embedding aprendido para cada usuário
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        # Embedding aprendido para categoria do item
        self.category_embedding = nn.Embedding(n_categories, embedding_dim)

        # Input: [user_emb | category_emb | available(1)]
        in_dim = embedding_dim * 2 + 1
        layers = []
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(0.2)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, user_ids, category_ids, available) -> torch.Tensor:
        u = self.user_embedding(user_ids)
        c = self.category_embedding(category_ids)
        x = torch.cat([u, c, available.unsqueeze(1).float()], dim=-1)
        return self.mlp(x).squeeze(-1)